# Tile and Trouble — Jane Street, May 2014

[https://www.janestreet.com/puzzles/tile-and-trouble-index/](https://www.janestreet.com/puzzles/tile-and-trouble-index/)

## Solution

         30 35 45 43 41 28 25 29 25 38 18 20
        ------------------------------------
     12|  1  2  2  1  0  1  0  0  2  2  0  1
     17|  0  2  2  0  0  3  3  3  2  2  0  0
     43|  5  5  5  5  5  3  3  3  0  3  3  3
     44|  5  5  5  5  5  3  3  3  1  3  3  3
     34|  5  5  5  5  5  0  0  0  0  3  3  3
     42|  5  5  5  5  5  1  4  4  4  4  0  0
     43|  5  5  5  5  5  1  4  4  4  4  0  1
     21|  2  2  0  1  0  0  4  4  4  4  0  0
     36|  2  2  4  4  4  4  4  4  4  4  0  0
     29|  0  0  4  4  4  4  0  2  2  3  3  3
     30|  0  1  4  4  4  4  0  2  2  3  3  3
     26|  0  1  4  4  4  4  0  0  0  3  3  3

### Answer: $20160$

## Goals

Continue to get better at using solvers.  This one seems well suited for *CP-SAT* so I'll continue to use that.

## AI use disclaimer

I used AI to help me write the code for this puzzle.

## The grid and the example

![Tile and Trouble grid](grid.png)

The example is a 7×7 grid whose answer is $$4 \times 2 \times 1^3 = 8$$.

In [9]:
from ortools.sat.python import cp_model

## The clues

We can sum the row clues (or the column clues) to give us the total number of points we need to place, which could help us a bit with bounding the problem.

In [ ]:
GRID_SIZE = 12

ROW_TOTALS = [12, 17, 43, 44, 34, 42, 43, 21, 36, 29, 30, 26]
COL_TOTALS = [30, 35, 45, 43, 41, 28, 25, 29, 25, 38, 18, 20]

EXAMPLE_SIZE = 7
EXAMPLE_ROW_TOTALS = [8, 8, 20, 17, 25, 25, 11]
EXAMPLE_COL_TOTALS = [15, 13, 10, 21, 20, 18, 17]


print("total points to place:", sum(ROW_TOTALS))

total points to place: 377


A total of 377 means we cannot have an 8x8x8s block, which was obvious from the row totals but still useful context.  Unfortunately, the existance of 1x1x1 tiles make the sum of cubes constraint somewhat irrelevant.  

## Helper functions

We can uniquely determine a tile from its top leftmost square.  We will write helper functions to find all possible placements for each tile size, as well as quickly lookup all the cells hit by each placement.  

In [ ]:
def row_total(grid, row):
    return sum(grid[row])


def column_total(grid, column):
    """Return the points in one column."""
    return sum(grid[i][column] for i in range(len(grid)))


def all_cells(size):
    """Return every (row, column) pair of a square grid, reading across then down."""
    cells = []
    for row in range(size):
        for column in range(size):
            cells.append((row, column))
    return cells


def all_placements(size):
    """Return every (row, column, tile_size) a square tile could legally occupy."""
    placements = []
    for tile_size in range(1, size + 1):
        for row in range(size - tile_size + 1):
            for column in range(size - tile_size + 1):
                placements.append((row, column, tile_size))
    return placements


def cells_of(placement):
    """Return every cell a placed tile covers."""
    top_row, left_column, tile_size = placement
    cells = []
    for row in range(top_row, top_row + tile_size):
        for column in range(left_column, left_column + tile_size):
            cells.append((row, column))
    return cells


def grid_from_tiles(tiles, size):
    """Return the grid of point values a list of tiles produces, with EMPTY where no tile sits."""
    grid = []
    for row in range(size):
        grid.append(0 * size)
    for placement in tiles:
        tile_size = placement[2]
        for row, column in cells_of(placement):
            grid[row][column] = tile_size
    return grid

In [ ]:
def problems_with(tiles, row_totals, col_totals):
    """Return a list of everything wrong with a candidate solution. Empty means it is legal."""
    size = len(row_totals)
    problems = []

    times_covered = {}
    for cell in all_cells(size):
        times_covered[cell] = 0

    for placement in tiles:
        top_row, left_column, tile_size = placement
        fits_on_the_board = (
            0 <= top_row
            and 0 <= left_column
            and top_row + tile_size <= size
            and left_column + tile_size <= size
        )
        if not fits_on_the_board:
            problems.append(f"tile {placement} runs off the board")
            continue
        for cell in cells_of(placement):
            times_covered[cell] += 1

    for cell in all_cells(size):
        if times_covered[cell] > 1:
            problems.append(f"cell {cell} is covered by {times_covered[cell]} tiles")

    grid = grid_from_tiles(tiles, size)
    for row in range(size):
        if row_total(grid, row) != row_totals[row]:
            problems.append(
                f"row {row} totals {row_total(grid, row)}, wanted {row_totals[row]}"
            )
    for column in range(size):
        if column_total(grid, column) != col_totals[column]:
            problems.append(
                f"column {column} totals {column_total(grid, column)}, wanted {col_totals[column]}"
            )

    return problems

In [4]:
def edge_neighbors(row, column, size):
    """Return the cells sharing an edge with this one. Diagonals are deliberately excluded."""
    neighbors = []
    for row_step, column_step in ((-1, 0), (1, 0), (0, -1), (0, 1)):
        neighbor_row = row + row_step
        neighbor_column = column + column_step
        row_on_board = 0 <= neighbor_row < size
        column_on_board = 0 <= neighbor_column < size
        if row_on_board and column_on_board:
            neighbors.append((neighbor_row, neighbor_column))
    return neighbors


def empty_regions(grid):
    """Return the area of each contiguous block of empty cells, largest first."""
    size = len(grid)
    already_seen = set()
    areas = []

    for start_cell in all_cells(size):
        start_row, start_column = start_cell
        if grid[start_row][start_column] != EMPTY:
            continue
        if start_cell in already_seen:
            continue

        # Flood fill outwards from this cell, collecting the whole connected blob.
        area = 0
        to_visit = [start_cell]
        already_seen.add(start_cell)
        while len(to_visit) > 0:
            row, column = to_visit.pop()
            area += 1
            for neighbor in edge_neighbors(row, column, size):
                neighbor_row, neighbor_column = neighbor
                is_empty = grid[neighbor_row][neighbor_column] == EMPTY
                if is_empty and neighbor not in already_seen:
                    already_seen.add(neighbor)
                    to_visit.append(neighbor)
        areas.append(area)

    areas.sort(reverse=True)
    return areas


def answer_for(grid):
    """Return the puzzle's answer: the product of the areas of the empty regions."""
    product = 1
    for area in empty_regions(grid):
        product = product * area
    return product

In [15]:
def show_grid(grid, row_totals, col_totals):
    """Print the grid with its row and column totals, using a 0 for empty cells."""
    size = len(grid)
    print("    " + " ".join(f"{col_totals[column]:>2}" for column in range(size)))
    print("    " + "---" * size)
    for row in range(size):
        cells = []
        for value in grid[row]:
            if value == 0:
                cells.append(" 0")
            else:
                cells.append(f"{value:>2}")
        print(f"{row_totals[row]:>3}| " + " ".join(cells))

## Validate on the published example

The example grid, read off the picture as a list of tiles. If `problems_with` finds nothing and
the answer comes out as 8, both the checker and my reading of the rules are right.

In [16]:
EXAMPLE_TILES = [
    (0, 0, 1),
    (0, 2, 1),
    (0, 3, 2),
    (0, 5, 1),
    (0, 6, 1),
    (1, 0, 2),
    (2, 3, 4),
    (3, 0, 1),
    (4, 0, 3),
    (6, 3, 1),
    (6, 5, 1),
]

example_problems = problems_with(EXAMPLE_TILES, EXAMPLE_ROW_TOTALS, EXAMPLE_COL_TOTALS)
assert len(example_problems) == 0, example_problems

example_grid = grid_from_tiles(EXAMPLE_TILES, EXAMPLE_SIZE)

# Areas 4, 2, 1, 1, 1 -- the page states the answer as 4 x 2 x 1^3 = 8.
assert empty_regions(example_grid) == [4, 2, 1, 1, 1], empty_regions(example_grid)
assert answer_for(example_grid) == 8

show_grid(example_grid, EXAMPLE_ROW_TOTALS, EXAMPLE_COL_TOTALS)
print()
print(
    "empty regions:", empty_regions(example_grid), "-> answer", answer_for(example_grid)
)

    15 13 10 21 20 18 17
    ---------------------
  8|  1  0  1  2  2  1  1
  8|  2  2  0  2  2  0  0
 20|  2  2  0  4  4  4  4
 17|  1  0  0  4  4  4  4
 25|  3  3  3  4  4  4  4
 25|  3  3  3  4  4  4  4
 11|  3  3  3  1  0  1  0

empty regions: [4, 2, 1, 1, 1] -> answer 8


## The model

We will build the model by assigning one boolean variable per cell per tile size that, if true, indicates a tile of that size has its top left corner on that cell.  The constraints would then be...

Variable: `is_placed[(row, column, tile_size)]` indicates a tile of tile_size with top left at row, column.

1. Propogated tiles fall within the grid
2. No cell is assigned to more than 1 tile
3. The row and column sums are correct after all placements --> Notice that a tiles contribution is always n^2 to all rows and columns it spans.


In [ ]:
def build_model(row_totals, col_totals):
    """Build the Tile and Trouble model.

    Returns (model, is_placed), where is_placed[(row, column, tile_size)] is true when a tile of
    that size sits with its top-left corner on that cell.
    """
    size = len(row_totals)
    model = cp_model.CpModel()
    placements = all_placements(size)  # Get every possible placement

    is_placed = {}
    for placement in placements:
        is_placed[placement] = model.new_bool_var(f"tile_{placement}")

    # Tiles may not overlap, so every cell lies inside at most one of them.
    placements_covering = {}
    for cell in all_cells(size):
        placements_covering[cell] = []  #
    for placement in placements:
        for cell in cells_of(placement):
            placements_covering[cell].append(placement)
    for cell in all_cells(size):
        model.add_at_most_one([is_placed[p] for p in placements_covering[cell]])

    # Rowwise constraints
    for row in range(size):
        contributions = []
        for placement in placements:
            top_row, left_column, tile_size = placement
            crosses_this_row = top_row <= row < top_row + tile_size
            if crosses_this_row:
                contributions.append(tile_size * tile_size * is_placed[placement])
        model.add(sum(contributions) == row_totals[row])

    for column in range(size):
        contributions = []
        for placement in placements:
            top_row, left_column, tile_size = placement
            crosses_this_column = left_column <= column < left_column + tile_size
            if crosses_this_column:
                contributions.append(tile_size * tile_size * is_placed[placement])
        model.add(sum(contributions) == col_totals[column])

    return model, is_placed


def tiles_from_solver(solver, is_placed):
    """Return the list of placements the solver chose."""
    tiles = []
    for placement, was_placed in is_placed.items():
        if solver.value(was_placed) == 1:
            tiles.append(placement)
    tiles.sort()
    return tiles

## Example grid

We will test our solver on the example grid.

In [17]:
example_model, example_is_placed = build_model(EXAMPLE_ROW_TOTALS, EXAMPLE_COL_TOTALS)

example_solver = cp_model.CpSolver()
example_solver.parameters.max_time_in_seconds = 60.0
example_solver.parameters.num_workers = 8
example_status = example_solver.solve(example_model)
print(example_solver.status_name(example_status))

solved_tiles = tiles_from_solver(example_solver, example_is_placed)
solved_grid = grid_from_tiles(solved_tiles, EXAMPLE_SIZE)

# Whatever it found has to be legal, whether or not it matches the published picture.
assert len(problems_with(solved_tiles, EXAMPLE_ROW_TOTALS, EXAMPLE_COL_TOTALS)) == 0

show_grid(solved_grid, EXAMPLE_ROW_TOTALS, EXAMPLE_COL_TOTALS)
print()
print("answer:", answer_for(solved_grid), "(the published example answer is 8)")

OPTIMAL
    15 13 10 21 20 18 17
    ---------------------
  8|  1  0  1  2  2  1  1
  8|  2  2  0  2  2  0  0
 20|  2  2  0  4  4  4  4
 17|  1  0  0  4  4  4  4
 25|  3  3  3  4  4  4  4
 25|  3  3  3  4  4  4  4
 11|  3  3  3  1  0  1  0

answer: 8 (the published example answer is 8)


## Solution

In [18]:
model, is_placed = build_model(ROW_TOTALS, COL_TOTALS)

solver = cp_model.CpSolver()
solver.parameters.max_time_in_seconds = 120.0
solver.parameters.num_workers = 8
solver.parameters.log_search_progress = True
status = solver.solve(model)
print(solver.status_name(status))


Starting CP-SAT solver v9.15.6755
Parameters: max_time_in_seconds: 120 log_search_progress: true num_workers: 8

Initial satisfaction model '': (model_fingerprint: 0x4668bbed1fb4d0ff)
#Variables: 650 (647 primary variables)
  - 650 Booleans in [0,1]
#kAtMostOne: 144 (#literals: 12'376)
#kLinearN: 24 (#terms: 4'732)

Starting presolve at 0.00s
  1.18e-04s  0.00e+00d  [DetectDominanceRelations] 
  1.58e-03s  0.00e+00d  [PresolveToFixPoint] #num_loops=2 #num_dual_strengthening=1 
  2.80e-05s  0.00e+00d  [ExtractEncodingFromLinear] #potential_supersets=144 
  8.50e-05s  0.00e+00d  [DetectDuplicateColumns] 
  7.00e-06s  0.00e+00d  [DetectDuplicateConstraints] 
[Symmetry] Graph for symmetry has 1'117 nodes and 5'137 arcs.
[Symmetry] Symmetry computation done. time: 0.000181 dtime: 0.00038963
  1.40e-05s  0.00e+00d  [DetectDuplicateConstraintsWithDifferentEnforcements] #without_enforcements=4 
  3.17e-03s  6.76e-03d  [Probe] #probed=878 #fixed_bools=8 #new_binary_clauses=1'467 
  5.31e-04s  

In [19]:
tiles = tiles_from_solver(solver, is_placed)
grid = grid_from_tiles(tiles, GRID_SIZE)

# Never trust the model over the checker.
problems = problems_with(tiles, ROW_TOTALS, COL_TOTALS)
assert len(problems) == 0, problems

show_grid(grid, ROW_TOTALS, COL_TOTALS)
print()
print("tiles placed:", len(tiles))
print("empty regions:", empty_regions(grid))
print("answer:", answer_for(grid))

    30 35 45 43 41 28 25 29 25 38 18 20
    ------------------------------------
 12|  1  2  2  1  0  1  0  0  2  2  0  1
 17|  0  2  2  0  0  3  3  3  2  2  0  0
 43|  5  5  5  5  5  3  3  3  0  3  3  3
 44|  5  5  5  5  5  3  3  3  1  3  3  3
 34|  5  5  5  5  5  0  0  0  0  3  3  3
 42|  5  5  5  5  5  1  4  4  4  4  0  0
 43|  5  5  5  5  5  1  4  4  4  4  0  1
 21|  2  2  0  1  0  0  4  4  4  4  0  0
 36|  2  2  4  4  4  4  4  4  4  4  0  0
 29|  0  0  4  4  4  4  0  2  2  3  3  3
 30|  0  1  4  4  4  4  0  2  2  3  3  3
 26|  0  1  4  4  4  4  0  0  0  3  3  3

tiles placed: 21
empty regions: [7, 5, 4, 4, 3, 3, 2, 2, 1, 1, 1]
answer: 20160


We found a valid solution in 0.36841 seconds -- showing that this problem is small enough that even our fully unconstrained model solves it quickly.  If it was slow, we could add specific knowledge (i.e. you can't have anything higher than 7 in the grid, column 1 cannot have anything higher than 3, etc...) to speed it up.  

## Confirm the solution is unique

There is no objective here, so `OPTIMAL` only means "a solution exists" — it says nothing about
whether another one does. Since the puzzle asks for *the* answer, that is worth checking:
enumerate every solution and confirm they all give the same product.

`enumerate_all_solutions` requires a single worker, so this is slower than the solve above.